In [10]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import networkx as nx
import community as community_louvain

from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
pd.set_option("display.max_columns", 100)

#### Применяем StandardScaler, чтобы привести все переменные в общий вид, также делаем некоторые проверки еще раз

In [11]:
df_unscaled = pd.read_csv("datasets/df_sberne.csv")
df = df_unscaled.copy()

cols = ['Средняя зарплата', 'population',
      'urban_share', 'share_industry', 'share_agriculture', 'share_services',
      'Здоровье_share', 'Маркетплейсы_share', 'Общественное питание_share',
    'Продовольствие_share', 'Транспорт_share']

scaler = StandardScaler().set_output(transform="pandas")
df[cols] = scaler.fit_transform(df[cols])

df['year']  = df['year'].astype(int)
df['month'] = df['month'].astype(int)

cats = ['Здоровье_share', 'Маркетплейсы_share',
        'Общественное питание_share', 'Продовольствие_share', 'Транспорт_share']

print("df:", df.shape)
print("Уникальные (year, month):")
print(df[['year', 'month']].drop_duplicates().sort_values(['year', 'month']).to_string(index=False))

df: (23915, 15)
Уникальные (year, month):
 year  month
 2023      1
 2023      2
 2023      3
 2023      4
 2023      5
 2023      6
 2023      7
 2023      8
 2023      9
 2023     10
 2023     11
 2023     12
 2024      1
 2024      2
 2024      3
 2024      4
 2024      5
 2024      6
 2024      7
 2024      8
 2024      9
 2024     10
 2024     11
 2024     12


## Начало эксперимента
Чтобы понять, какой способ проведения ребер лучше, проведем эксперимент - применим кластеризацию с каждым из методов для всех 24 методов и усредним метрики IVCI по ним, а затем решим какой из них лучше. Также переберем параметр resolution в louvain-кластеризации.

In [12]:
feature_cols = [
    'Здоровье_share',
    'Маркетплейсы_share',
    'Общественное питание_share',
    'Продовольствие_share',
    'Транспорт_share',
    'share_industry',
    'share_agriculture',
    'share_services',
    'urban_share',
    'Средняя зарплата',
    'population',
]

PERCENTILE_SIM = 75
K_MUTUAL       = 250
RESOLUTIONS    = [0.9, 1.0, 1.1]
TANIMOTO_EPS   = 1e-12

global_feature_min = (
    df[feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .min()
    .values
    .astype(float)
)

PERIODS = sorted(df[['year', 'month']].drop_duplicates().itertuples(index=False, name=None))
print(f"Всего периодов: {len(PERIODS)}")
print(PERIODS)

Всего периодов: 24
[(2023, 1), (2023, 2), (2023, 3), (2023, 4), (2023, 5), (2023, 6), (2023, 7), (2023, 8), (2023, 9), (2023, 10), (2023, 11), (2023, 12), (2024, 1), (2024, 2), (2024, 3), (2024, 4), (2024, 5), (2024, 6), (2024, 7), (2024, 8), (2024, 9), (2024, 10), (2024, 11), (2024, 12)]


Для сравнения выберем 4 наиболее перспективных метода:
- cosine similarity
- adaptive rbf
- tanimoto similarity
- mutual knn

Будем проводить топ-25% ребер, а в knn для честного сравнения будем делать 250 ближайших соседей.

In [13]:
def sim_cosine(X):
    S = cosine_similarity(X)
    np.fill_diagonal(S, 0)
    return S


def tanimoto_similarity(X):
    Xp = np.asarray(X, dtype=float)
    dot = Xp @ Xp.T
    sq = np.sum(Xp**2, axis=1)
    denom = sq[:, None] + sq[None, :] - dot
    S = np.divide(dot, denom, out=np.zeros_like(dot), where=denom > 0)
    np.fill_diagonal(S, 0)
    return S


def adaptive_rbf_similarity(X, k_local=7):
    n = X.shape[0]
    nn = NearestNeighbors(n_neighbors=min(k_local + 1, n)).fit(X)
    dist, _ = nn.kneighbors(X)
    sigma = np.maximum(dist[:, -1], 1e-9)
    D2 = euclidean_distances(X, squared=True)
    denom = np.outer(sigma, sigma)
    W = np.exp(-D2 / denom)
    np.fill_diagonal(W, 0)
    return W


In [14]:
def build_threshold_graph_percentile_sim(sim, percentile=75):
    n = sim.shape[0]
    S = sim.copy()
    np.fill_diagonal(S, 0)

    iu = np.triu_indices_from(S, k=1)
    vals = S[iu]
    pos = vals[vals > 0]
    if len(pos) == 0:
        return nx.Graph(), np.nan
    thr = np.percentile(pos, percentile)

    A = np.zeros((n, n), dtype=bool)
    A[iu] = vals >= thr
    A = A | A.T

    G = nx.Graph()
    G.add_nodes_from(range(n))
    ii, jj = np.where(np.triu(A, k=1))
    for i, j in zip(ii, jj):
        G.add_edge(int(i), int(j), weight=float(S[i, j]))
    return G, float(thr)


def build_mutual_knn_graph(X, k=250):
    n = X.shape[0]
    k_eff = min(k, n - 1)
    nn = NearestNeighbors(n_neighbors=k_eff + 1).fit(X)
    dist, idx = nn.kneighbors(X)
    idx = idx[:, 1:]
    dist = dist[:, 1:]

    neighbor_sets = [set(idx[i].tolist()) for i in range(n)]
    sigma = np.maximum(dist[:, -1], 1e-9)

    G = nx.Graph()
    G.add_nodes_from(range(n))
    for i in range(n):
        for j in neighbor_sets[i]:
            if i < j and i in neighbor_sets[j]:
                d2 = float(np.sum((X[i] - X[j]) ** 2))
                w = float(np.exp(-d2 / (sigma[i] * sigma[j] + 1e-12)))
                G.add_edge(i, j, weight=w)

    for i in range(n):
        if G.degree(i) == 0:
            j = int(idx[i, 0])
            d2 = float(np.sum((X[i] - X[j]) ** 2))
            w = float(np.exp(-d2 / (sigma[i] * sigma[j] + 1e-12)))
            G.add_edge(i, j, weight=w)
    return G


def graph_stats(G):
    n = G.number_of_nodes()
    m = G.number_of_edges()
    return {"n_nodes": n, "n_edges": m,
            "avg_degree": round(2 * m / n, 2) if n > 0 else 0}

In [15]:
def louvain_partition(G, seed=42, resolution=1.0):
    if G.number_of_nodes() == 0:
        return np.array([])
    part = community_louvain.best_partition(
        G, weight='weight', random_state=seed, resolution=resolution
    )
    n = G.number_of_nodes()
    return np.array([part[i] for i in range(n)])

def cluster_sizes(labels):
    return pd.Series(labels).value_counts().sort_index()

#### Задаем IVCI-метрики

In [16]:
def s_dbw(X, labels):
    labs = np.unique(labels)
    k = len(labs)
    if k < 2:
        return np.nan

    centroids = np.array([X[labels == lab].mean(axis=0) for lab in labs])
    cluster_std = np.array([
        np.std(X[labels == lab], axis=0, ddof=0)
        for lab in labs
    ])
    global_std = np.std(X, axis=0, ddof=0)

    global_std_norm = np.linalg.norm(global_std)
    if global_std_norm <= 0:
        return np.nan

    scat = np.mean(np.linalg.norm(cluster_std, axis=1)) / global_std_norm
    stdev = global_std_norm

    def density(center, indices):
        if len(indices) == 0:
            return 0
        d = np.linalg.norm(X[indices] - center, axis=1)
        return int(np.sum(d <= stdev))

    dens_centers = np.array([
        density(centroids[i], np.where(labels == labs[i])[0])
        for i in range(k)
    ], dtype=float)

    dens_bw = 0.0
    pair_count = 0

    for i in range(k):
        idx_i = np.where(labels == labs[i])[0]
        for j in range(i + 1, k):
            idx_j = np.where(labels == labs[j])[0]

            midpoint = 0.5 * (centroids[i] + centroids[j])
            dens_mid = density(midpoint, np.concatenate([idx_i, idx_j]))

            denom = max(dens_centers[i], dens_centers[j])
            if denom > 0:
                dens_bw += dens_mid / denom

            pair_count += 1

    if pair_count > 0:
        dens_bw /= pair_count

    return float(scat + dens_bw)


def avi_score(G, partition_dict):
    nodes = list(G.nodes())
    if len(nodes) == 0:
        return np.nan

    A = nx.to_numpy_array(G, nodelist=nodes, weight="weight", dtype=float)
    labels = np.array([partition_dict[n] for n in nodes])
    labs = np.unique(labels)

    values = []
    for lab in labs:
        mask = labels == lab
        internal = A[np.ix_(mask, mask)].sum()
        external = A[np.ix_(mask, ~mask)].sum()
        denom = internal + external
        values.append(internal / denom if denom > 0 else np.nan)

    return float(np.nanmean(values))


def avu_score(G, partition_dict):
    nodes = list(G.nodes())
    if len(nodes) == 0:
        return np.nan

    A = nx.to_numpy_array(G, nodelist=nodes, weight="weight", dtype=float)
    labels = np.array([partition_dict[n] for n in nodes])
    labs = np.unique(labels)
    k = len(labs)

    if k < 2:
        return np.nan

    ext = {}
    for lab in labs:
        mask = labels == lab
        ext[lab] = A[np.ix_(mask, ~mask)].sum()

    total = 0.0
    for lab_k in labs:
        mask_k = labels == lab_k
        for lab_l in labs:
            if lab_l == lab_k:
                continue

            mask_l = labels == lab_l
            between = A[np.ix_(mask_k, mask_l)].sum()
            denom = ext[lab_k] + ext[lab_l] - between

            if denom > 0:
                total += between / denom

    return float(total / k)


def mq_score(G, partition_dict):
    nodes = list(G.nodes())
    if len(nodes) == 0:
        return np.nan

    A = nx.to_numpy_array(G, nodelist=nodes, weight="weight", dtype=float)
    labels = np.array([partition_dict[n] for n in nodes])
    labs = np.unique(labels)
    k = len(labs)

    if k == 1:
        mask = labels == labs[0]
        n = int(mask.sum())
        if n < 2:
            return np.nan
        internal = A[np.ix_(mask, mask)].sum() / 2.0
        return float(internal / (n * (n - 1) / 2.0))

    intra = []
    for lab in labs:
        mask = labels == lab
        n_i = int(mask.sum())
        if n_i < 2:
            intra.append(0.0)
            continue

        internal = A[np.ix_(mask, mask)].sum() / 2.0
        max_internal = n_i * (n_i - 1) / 2.0
        intra.append(internal / max_internal)

    inter = []
    for i, lab_i in enumerate(labs):
        mask_i = labels == lab_i
        n_i = int(mask_i.sum())

        for lab_j in labs[i + 1:]:
            mask_j = labels == lab_j
            n_j = int(mask_j.sum())

            between = A[np.ix_(mask_i, mask_j)].sum()
            max_between = n_i * n_j

            inter.append(
                between / max_between if max_between > 0 else 0.0
            )

    mean_inter = float(np.mean(inter)) if inter else 0.0
    return float(np.mean(intra) - mean_inter)


def evaluate_clustering_graph(G, X_eval, partition_dict, name="", n_obs=None):
    nodes = list(G.nodes())
    labels = np.array([partition_dict[n] for n in nodes])
    n_lab = len(np.unique(labels))

    if n_obs is None:
        n_obs = len(X_eval)

    res = {"method": name, "n_clusters": n_lab}

    if n_lab < 2:
        return {
            **res,
            "SW": np.nan,
            "CH": np.nan,
            "S_Dbw": np.nan,
            "AVI": np.nan,
            "AVU": np.nan,
            "MQ": np.nan,
        }

    res["SW"] = silhouette_score(X_eval, labels)

    res["CH"] = calinski_harabasz_score(X_eval, labels) / n_obs

    res["S_Dbw"] = s_dbw(X_eval, labels)
    res["AVI"] = avi_score(G, partition_dict)
    res["AVU"] = avu_score(G, partition_dict)
    res["MQ"] = mq_score(G, partition_dict)

    return res


In [17]:
def process_period(year, month):
    sub = df[(df['year'] == year) & (df['month'] == month)].copy()
    sub = sub.dropna(subset=feature_cols)
    sub['urban_share'] = sub['urban_share'].replace([np.inf, -np.inf], 0)
    sub = sub.reset_index(drop=True)

    n_obs = len(sub)
    if n_obs < 10:
        return []

    X = sub[feature_cols].values.astype(float)
    X_tan = X - global_feature_min + TANIMOTO_EPS

    S_cos   = sim_cosine(X)
    S_tan   = tanimoto_similarity(X_tan)
    S_adrbf = adaptive_rbf_similarity(X, k_local=7)

    G_cos, _ = build_threshold_graph_percentile_sim(
        S_cos, percentile=PERCENTILE_SIM
    )
    G_tan, _ = build_threshold_graph_percentile_sim(
        S_tan, percentile=PERCENTILE_SIM
    )
    G_ar, _ = build_threshold_graph_percentile_sim(
        S_adrbf, percentile=PERCENTILE_SIM
    )
    G_mut = build_mutual_knn_graph(X, k=K_MUTUAL)

    GRAPHS_LOCAL = {
        "cosine":       G_cos,
        "tanimoto":     G_tan,
        "mutual_knn":   G_mut,
        "adaptive_rbf": G_ar,
    }

    rows_local = []

    for method, G in GRAPHS_LOCAL.items():
        for res in RESOLUTIONS:
            part = community_louvain.best_partition(
                G,
                weight='weight',
                random_state=42,
                resolution=res
            )

            row = evaluate_clustering_graph(
                G, X, part, method, n_obs=n_obs
            )
            row["resolution"] = res
            row["year"] = year
            row["month"] = month
            row["n_obs"] = n_obs
            rows_local.append(row)

    return rows_local


#### Сам эксперимент

In [18]:
all_rows = []
for (y, m) in PERIODS:
    rows = process_period(y, m)
    all_rows.extend(rows)
    print(f"  {y}-{m:02d}: {len(rows)} строк, "
          f"n_obs={rows[0]['n_obs'] if rows else 0}")

raw_df = pd.DataFrame(all_rows)
print("\nraw_df:", raw_df.shape)
raw_df.head()

  2023-01: 12 строк, n_obs=1027
  2023-02: 12 строк, n_obs=1027
  2023-03: 12 строк, n_obs=1027
  2023-04: 12 строк, n_obs=1021
  2023-05: 12 строк, n_obs=1019
  2023-06: 12 строк, n_obs=1020
  2023-07: 12 строк, n_obs=1020
  2023-08: 12 строк, n_obs=1018
  2023-09: 12 строк, n_obs=1015
  2023-10: 12 строк, n_obs=1015
  2023-11: 12 строк, n_obs=1012
  2023-12: 12 строк, n_obs=1014
  2024-01: 12 строк, n_obs=1010
  2024-02: 12 строк, n_obs=1012
  2024-03: 12 строк, n_obs=1011
  2024-04: 12 строк, n_obs=961
  2024-05: 12 строк, n_obs=961
  2024-06: 12 строк, n_obs=958
  2024-07: 12 строк, n_obs=961
  2024-08: 12 строк, n_obs=963
  2024-09: 12 строк, n_obs=966
  2024-10: 12 строк, n_obs=956
  2024-11: 12 строк, n_obs=958
  2024-12: 12 строк, n_obs=963

raw_df: (288, 12)


,method,n_clusters,SW,CH,S_Dbw,AVI,AVU,MQ,resolution,year,month,n_obs
0,cosine,6,0.121898,0.099807,1.890164,0.496065,0.400107,0.338313,0.9,2023,1,1027
1,cosine,5,0.152417,0.142818,2.066659,0.597541,0.496066,0.311659,1.0,2023,1,1027
2,cosine,5,0.153096,0.144239,2.099472,0.599132,0.509906,0.307215,1.1,2023,1,1027
3,tanimoto,17,0.123179,0.051252,0.954952,0.534656,0.133574,0.343619,0.9,2023,1,1027
4,tanimoto,13,0.122101,0.051467,0.913547,0.645783,0.153935,0.220601,1.0,2023,1,1027


In [19]:
metric_cols = ["SW", "CH", "S_Dbw", "AVI", "AVU", "MQ", "n_clusters"]

agg_df = (
    raw_df
    .groupby(["method", "resolution"])[metric_cols]
    .mean()
    .reset_index()
    .sort_values(["method", "resolution"])
)

In [25]:
from IPython.display import display, HTML

DIRECTIONS = {
    "SW":    True,
    "CH":    True,
    "S_Dbw": False,
    "AVI":   True,
    "AVU":   False,
    "MQ":    True,
}

C1, BG1 = "#7ee787", "#0f3d1a"
C2, BG2 = "#ffd33d", "#4a3a00"
C3, BG3 = "#ff7b72", "#4a1512"

def color_top3(series, higher_is_better=True, decimals=4):
    s = series.copy()
    order = s.sort_values(ascending=not higher_is_better).index
    top3 = list(order[:3])

    colors = [
        (C1, BG1),
        (C2, BG2),
        (C3, BG3),
    ]

    out = pd.Series("", index=s.index)
    for idx in s.index:
        val = f"{s.loc[idx]:.{decimals}f}"
        if idx in top3:
            pos = top3.index(idx)
            c, bg = colors[pos]
            out.loc[idx] = (
                f'<span style="color:{c}; background:{bg}; '
                f'font-weight:bold; padding:2px 6px; border-radius:4px;">'
                f'{val}</span>'
            )
        else:
            out.loc[idx] = val
    return out


tab = agg_df.copy()

formatted = pd.DataFrame()
formatted["method"]     = tab["method"]
formatted["resolution"] = tab["resolution"]
formatted["k"]          = tab["n_clusters"].round(2)

for metric, higher_better in DIRECTIONS.items():
    formatted[metric] = color_top3(tab[metric], higher_is_better=higher_better)

html = formatted.to_html(index=False, escape=False, justify="center")

html = html.replace(
    '<table border="1" class="dataframe">',
    '<table style="border-collapse:collapse; text-align:center; '
    'font-family:Arial; font-size:13px; '
    'background:#1e1e1e; color:#e6e6e6;" border="1">'
)
html = html.replace(
    '<th>',
    '<th style="padding:7px 12px; background:#333333; color:#ffffff; '
    'border:1px solid #555; font-weight:bold;">'
)
html = html.replace(
    '<td>',
    '<td style="padding:5px 10px; border:1px solid #444; color:#e6e6e6;">'
)

legend = (
    '<div style="font-family:Arial; font-size:13px; margin:8px 0; color:#e6e6e6;">'
    f'<span style="background:{BG1}; color:{C1}; font-weight:bold; '
    'padding:3px 8px; border-radius:4px;">топ-1</span> &nbsp; '
    f'<span style="background:{BG2}; color:{C2}; font-weight:bold; '
    'padding:3px 8px; border-radius:4px;">топ-2</span> &nbsp; '
    f'<span style="background:{BG3}; color:{C3}; font-weight:bold; '
    'padding:3px 8px; border-radius:4px;">топ-3</span>'
    '</div>'
)

display(HTML(legend + html))

method,resolution,k,SW,CH,S_Dbw,AVI,AVU,MQ
adaptive_rbf,0.9,6.50,0.1408,0.1606,1.8099,0.6322,0.4933,0.1340
adaptive_rbf,1.0,5.50,0.1323,0.1623,2.0307,0.6307,0.5244,0.0824
adaptive_rbf,1.1,6.75,0.1118,0.1467,1.9867,0.5799,0.5061,0.0943
cosine,0.9,4.92,0.1544,0.1641,2.0526,0.6727,0.5854,0.3000
cosine,1.0,4.25,0.1485,0.1711,2.1574,0.7122,0.6274,0.2801
cosine,1.1,4.71,0.1488,0.1700,2.1638,0.6879,0.6137,0.2918
mutual_knn,0.9,4.83,0.1461,0.1632,1.9988,0.6536,0.5403,0.2446
mutual_knn,1.0,4.29,0.1345,0.1675,2.1677,0.6731,0.5895,0.2122
mutual_knn,1.1,4.79,0.1225,0.1585,2.1405,0.6335,0.5650,0.2217
tanimoto,0.9,8.83,0.1491,0.0896,0.8812,0.6355,0.2581,0.3687


Видим, что Tanimoto выигрывает по многим метрикам, однако он строит много кластеров, а это экономически не интерпретируемо. Так что ради интерпретируемости мы пожертвуем качеством и не будем использовать Tanimoto.

In [39]:
from IPython.display import display, HTML

DIRECTIONS = {
    "SW":    True,
    "CH":    True,
    "S_Dbw": False,
    "AVI":   True,
    "AVU":   False,
    "MQ":    True,
}

C1, BG1 = "#7ee787", "#0f3d1a"
C2, BG2 = "#ffd33d", "#4a3a00"
C3, BG3 = "#ff7b72", "#4a1512"

def color_top3(series, higher_is_better=True, decimals=4):
    s = series.copy()
    order = s.sort_values(ascending=not higher_is_better).index
    top3 = list(order[:3])

    colors = [
        (C1, BG1),
        (C2, BG2),
        (C3, BG3),
    ]

    out = pd.Series("", index=s.index)

    for idx in s.index:
        val = f'{s.loc[idx]:.{decimals}f}'

        if idx in top3:
            pos = top3.index(idx)
            c, bg = colors[pos]

            out.loc[idx] = (
                f'<span style="color:{c}; background:{bg}; '
                f'font-weight:bold; padding:2px 6px; border-radius:4px;">'
                f'{val}</span>'
            )
        else:
            out.loc[idx] = val

    return out


tab = agg_df[agg_df["method"] != "tanimoto"].copy()

formatted = pd.DataFrame()
formatted["method"]     = tab["method"]
formatted["resolution"] = tab["resolution"]
formatted["k"]          = tab["n_clusters"].round(2)

for metric, higher_better in DIRECTIONS.items():
    formatted[metric] = color_top3(
        tab[metric],
        higher_is_better=higher_better
    )

html = formatted.to_html(
    index=False,
    escape=False,
    justify="center"
)

html = html.replace(
    '<table border="1" class="dataframe">',
    '<table style="border-collapse:collapse; text-align:center; '
    'font-family:Arial; font-size:13px; '
    'background:#1e1e1e; color:#e6e6e6;" border="1">'
)

html = html.replace(
    '<th>',
    '<th style="padding:7px 12px; background:#333333; color:#ffffff; '
    'border:1px solid #555; font-weight:bold;">'
)

html = html.replace(
    '<td>',
    '<td style="padding:5px 10px; border:1px solid #444; color:#e6e6e6;">'
)

legend = (
    '<div style="font-family:Arial; font-size:13px; margin:8px 0; color:#e6e6e6;">'
    f'<span style="background:{BG1}; color:{C1}; font-weight:bold; '
    'padding:3px 8px; border-radius:4px;">топ-1</span> &nbsp; '
    f'<span style="background:{BG2}; color:{C2}; font-weight:bold; '
    'padding:3px 8px; border-radius:4px;">топ-2</span> &nbsp; '
    f'<span style="background:{BG3}; color:{C3}; font-weight:bold; '
    'padding:3px 8px; border-radius:4px;">топ-3</span>'
    '</div>'
)

display(HTML(legend + html))

method,resolution,k,SW,CH,S_Dbw,AVI,AVU,MQ
adaptive_rbf,0.9,6.50,0.1408,0.1606,1.8099,0.6322,0.4933,0.1340
adaptive_rbf,1.0,5.50,0.1323,0.1623,2.0307,0.6307,0.5244,0.0824
adaptive_rbf,1.1,6.75,0.1118,0.1467,1.9867,0.5799,0.5061,0.0943
cosine,0.9,4.92,0.1544,0.1641,2.0526,0.6727,0.5854,0.3000
cosine,1.0,4.25,0.1485,0.1711,2.1574,0.7122,0.6274,0.2801
cosine,1.1,4.71,0.1488,0.1700,2.1638,0.6879,0.6137,0.2918
mutual_knn,0.9,4.83,0.1461,0.1632,1.9988,0.6536,0.5403,0.2446
mutual_knn,1.0,4.29,0.1345,0.1675,2.1677,0.6731,0.5895,0.2122
mutual_knn,1.1,4.79,0.1225,0.1585,2.1405,0.6335,0.5650,0.2217


In [40]:
tab = agg_df[agg_df["method"] != "tanimoto"].copy()

Смотрим на адекватность размеров кластеров:

### Смотрим на корелляции между метриками

In [41]:
tab = tab[tab.method!="tanimoto"]

In [42]:
columns = ['method', 'resolution', 'SW', 'CH', 'S_Dbw', 'AVI', 'AVU', 'MQ']

metrics = ['SW', 'CH', 'S_Dbw', 'AVI', 'AVU', 'MQ']

corr_spearman = tab[metrics].corr(method='spearman')

corr_spearman.round(3)

,SW,CH,S_Dbw,AVI,AVU,MQ
SW,1.000,0.767,0.300,0.733,0.567,0.850
CH,0.767,1.000,0.717,0.933,0.867,0.650
S_Dbw,0.300,0.717,1.000,0.783,0.900,0.467
AVI,0.733,0.933,0.783,1.000,0.933,0.783
AVU,0.567,0.867,0.900,0.933,1.000,0.717
MQ,0.850,0.650,0.467,0.783,0.717,1.000


### Корелляция очень большая, поэтому странно было бы учитывать их все на равных
#### Используем метод CRITIC, который учитывает различность метрик.

In [43]:
metrics = [
    'SW',
    'CH',
    'S_Dbw',
    'AVI',
    'AVU',
    'MQ'
]

higher_is_better = {
    'SW': True,
    'CH': True,
    'S_Dbw': False,
    'AVI': True,
    'AVU': False,
    'MQ': True
}

In [44]:
norm_df = pd.DataFrame(index=tab.index)

for metric in metrics:
    x = tab[metric]

    if higher_is_better[metric]:
        norm_df[metric] = (x - x.min()) / (x.max() - x.min())
    else:
        norm_df[metric] = (x.max() - x) / (x.max() - x.min())

In [45]:
std = norm_df.std()

corr = norm_df.corr(method='spearman')

In [46]:
critic_score = {}

for metric in metrics:
    conflict = sum(
        1 - corr.loc[metric, other]
        for other in metrics
        if other != metric
    )

    critic_score[metric] = std[metric] * conflict

critic_score = pd.Series(critic_score)

weights = critic_score / critic_score.sum()

print("Веса метрик:")
print(weights.sort_values(ascending=False).round(3))

Веса метрик:
AVU      0.259
S_Dbw    0.214
MQ       0.153
CH       0.129
AVI      0.128
SW       0.117
dtype: float64


In [47]:
borda = pd.DataFrame(index=tab.index)

n = len(tab)

for metric in metrics:

    if higher_is_better[metric]:
        rank = tab[metric].rank(
            ascending=False,
            method='average'
        )
    else:
        rank = tab[metric].rank(
            ascending=True,
            method='average'
        )

    borda[metric] = n - rank

In [48]:
weighted_borda = pd.Series(0.0, index=tab.index)

for metric in metrics:
    weighted_borda += borda[metric] * weights[metric]

In [49]:
ranking = tab[
    ['method', 'resolution']
].copy()

ranking['Borda_score'] = weighted_borda

ranking = ranking.sort_values(
    'Borda_score',
    ascending=False
).reset_index(drop=True)

ranking

,method,resolution,Borda_score
0,cosine,0.9,5.079065
1,adaptive_rbf,0.9,5.068545
2,mutual_knn,0.9,4.956219
3,cosine,1.1,4.164959
4,cosine,1.0,4.106639
5,adaptive_rbf,1.1,3.460979
6,adaptive_rbf,1.0,3.370495
7,mutual_knn,1.1,2.920734
8,mutual_knn,1.0,2.872365


### Смотрим на адекватность размеров кластеров

In [50]:
BEST_METHOD = "cosine"
BEST_RESOLUTION = 0.9

cluster_sizes_by_month = {}
rows = []

for (y, m) in PERIODS:
    sub = df[(df['year'] == y) & (df['month'] == m)].copy()
    sub = sub.dropna(subset=feature_cols)
    sub['urban_share'] = sub['urban_share'].replace([np.inf, -np.inf], 0)
    sub = sub.reset_index(drop=True)

    if len(sub) < 10:
        continue

    X_m = sub[feature_cols].values.astype(float)

    S_cos = sim_cosine(X_m)
    G, _ = build_threshold_graph_percentile_sim(
        S_cos, percentile=PERCENTILE_SIM
    )

    part = community_louvain.best_partition(
        G,
        weight='weight',
        random_state=42,
        resolution=BEST_RESOLUTION,
    )

    sizes = (
        pd.Series(list(part.values()))
        .value_counts()
        .sort_values(ascending=False)
        .tolist()
    )

    cluster_sizes_by_month[(y, m)] = sizes

    rows.append({
        "year": y,
        "month": m,
        "n_obs": len(sub),
        "n_clusters": len(sizes),
        "min_size": min(sizes),
        "max_size": max(sizes),
        "median_size": int(np.median(sizes)),
        "sizes": sizes,
    })

sizes_df = pd.DataFrame(rows)

pd.set_option("display.max_colwidth", None)
for _, r in sizes_df.iterrows():
    print(f"{int(r['year'])}-{int(r['month']):02d}  "
          f"n_obs={r['n_obs']:4d}  k={r['n_clusters']:2d}  "
          f"sizes={r['sizes']}")

2023-01  n_obs=1027  k= 6  sizes=[354, 343, 257, 43, 17, 13]
2023-02  n_obs=1027  k= 4  sizes=[350, 338, 244, 95]
2023-03  n_obs=1027  k= 5  sizes=[329, 294, 232, 116, 56]
2023-04  n_obs=1021  k= 5  sizes=[343, 240, 229, 134, 75]
2023-05  n_obs=1019  k= 4  sizes=[361, 310, 223, 125]
2023-06  n_obs=1020  k= 4  sizes=[357, 336, 232, 95]
2023-07  n_obs=1020  k= 5  sizes=[351, 317, 227, 73, 52]
2023-08  n_obs=1018  k= 5  sizes=[342, 277, 232, 121, 46]
2023-09  n_obs=1015  k= 5  sizes=[356, 316, 225, 59, 59]
2023-10  n_obs=1015  k= 5  sizes=[311, 285, 233, 100, 86]
2023-11  n_obs=1012  k= 5  sizes=[342, 217, 212, 167, 74]
2023-12  n_obs=1014  k= 4  sizes=[332, 301, 234, 147]
2024-01  n_obs=1010  k= 5  sizes=[315, 256, 232, 178, 29]
2024-02  n_obs=1012  k= 6  sizes=[313, 223, 210, 170, 82, 14]
2024-03  n_obs=1011  k= 6  sizes=[311, 216, 211, 154, 102, 17]
2024-04  n_obs= 961  k= 5  sizes=[279, 230, 193, 153, 106]
2024-05  n_obs= 961  k= 4  sizes=[324, 293, 202, 142]
2024-06  n_obs= 958  k= 5